# 03 — Static GCN classifier

This implements the static graph stage. A train-only label co-occurrence graph generates one classifier vector per label through GCN layers. The backbone can be initialised from the baseline checkpoint from notebook 02.

In [ ]:

from pathlib import Path
import sys, os, json, time

# Locate the project root whether this folder is in /kaggle/working or uploaded as a Kaggle dataset.
candidates = [Path.cwd(), Path('/kaggle/working/ladi_v2_gcn_project')]
if Path('/kaggle/input').exists():
    candidates += list(Path('/kaggle/input').glob('*'))
PROJECT_ROOT = None
for c in candidates:
    if (c / 'src' / 'ladi_config.py').exists():
        PROJECT_ROOT = c
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find project src/. Upload/unzip the full ladi_v2_gcn_project folder first.')
sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
from pathlib import Path
import torch

from src.ladi_config import LABEL_COLS_V2A
from src.ladi_data import make_loaders
from src.ladi_models import StaticGCNClassifier, init_classifier_bias, pos_weight_from_df, load_backbone_from_checkpoint
from src.ladi_train import seed_everything, train_model

seed_everything(42)
CACHE_DIR = Path('/kaggle/working/ladi_v2a_cache')
OUT_DIR = Path('/kaggle/working/ladi_v2_outputs')
RUN_DIR = OUT_DIR / 'runs'
GRAPH_DIR = OUT_DIR / 'graphs'

BACKBONE = 'resnet18'
IMG_SIZE = 320
BATCH_SIZE = 16
EPOCHS = 8
LR = 2e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
AMP = True
GRAPH_MODE = 'pmi'  # pmi or condprob; pmi is a strong default for positive association.
BASELINE_CKPT = RUN_DIR / f'baseline_{BACKBONE}_{IMG_SIZE}' / 'best_model.pt'


In [ ]:
train_loader, val_loader, test_loader, label_cols, train_df, val_df, test_df = make_loaders(
    CACHE_DIR, label_cols=LABEL_COLS_V2A, img_size=IMG_SIZE, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
)
train_freq = train_df[label_cols].mean().values
pos_weight = pos_weight_from_df(train_df, label_cols, max_weight=8.0)
A = torch.load(GRAPH_DIR / f'A_{GRAPH_MODE}.pt')
print('Loaded graph:', GRAPH_MODE, A.shape)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = StaticGCNClassifier(
    num_labels=len(label_cols),
    A_norm=A,
    backbone=BACKBONE,
    pretrained=True,
    label_dim=256,
    gcn_hidden=512,
    dropout=0.25,
)
init_classifier_bias(model, train_freq)
if BASELINE_CKPT.exists():
    load_backbone_from_checkpoint(model, str(BASELINE_CKPT), device)
else:
    print('Baseline checkpoint not found; training with torchvision initialisation only.')

metrics, ckpt_path = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device,
    out_dir=RUN_DIR,
    run_name=f'static_gcn_{GRAPH_MODE}_{BACKBONE}_{IMG_SIZE}',
    label_cols=label_cols,
    train_freq=train_freq,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    pos_weight=pos_weight,
    amp=AMP,
    freeze_backbone_epochs=1,
)
print('Best checkpoint:', ckpt_path)
metrics['test_at_0_5']


For ablation, rerun this notebook with `GRAPH_MODE = "identity"` or `"condprob"`. Identity shows whether the graph structure adds value beyond extra parameters.